<a href="https://colab.research.google.com/github/russellfloyddm/pse-trading-bot/blob/main/ai_model/transformer_next_steps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PSE Trading Bot — Transformer Signal Model: End-to-End Training Notebook

> **✅ Colab-ready** — Run on [Google Colab](https://colab.research.google.com/) or locally. Before training, select a GPU runtime: *Runtime → Change runtime type → T4 GPU*.

This notebook walks you through every step in `ai_model/NEXT_STEPS.md`, ready for immediate execution:

| # | Section |
|---|---------|
| 1 | ⚙️  Configuration (tickers, dates, hyperparameters) |
| 2 | 📥  Data collection with **yfinance** |
| 3 | 🔍  Data quality checks |
| 4 | 📊  Price visualisation |
| 5 | 🔧  Feature engineering |
| 6 | 🏷️  Label design & class-imbalance analysis |
| 7 | 🗂️  Dataset construction |
| 8 | 🏗️  Transformer model architecture |
| 9 | 🚀  Training |
| 10 | 📈  Training curves |
| 11 | 🎯  Evaluation (classification + trading metrics) |
| 12 | 🤝  Trading-agent integration |
| 13 | 🔬  Hyperparameter tuning template |
| 14 | 🚢  Deployment (TorchScript export) |

> **Quick start:** Run the **🌐 Colab / Local Setup** cells at the top first, then work through Sections 0–14 in order. Only edit **Section 1** to change tickers or dates.

## 🌐 Colab / Local Setup

**Run this cell first** before anything else.  It:

1. Detects whether the notebook is running in **Google Colab** or locally.
2. **Clones the project repository** into `/content/pse-trading-bot` (Colab only).
3. Adds the project root to `sys.path` so all local modules are importable.  

> **Colab GPU tip:** before running training, go to *Runtime → Change runtime type* and select **T4 GPU** (free tier).

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# 🌐 COLAB / LOCAL ENVIRONMENT SETUP — run this cell first
# ─────────────────────────────────────────────────────────────────────────────
import sys
import os
import subprocess

# Detect Google Colab
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Clone or update the project repository ──────────────────────────────
    REPO_URL = "https://github.com/russellfloyddm/pse-trading-bot.git"
    REPO_DIR = "/content/pse-trading-bot"

    _is_valid_repo = os.path.isdir(os.path.join(REPO_DIR, ".git"))
    if not _is_valid_repo:
        if os.path.isdir(REPO_DIR):
            print("⚠️  Directory exists but is not a git repository — removing and re-cloning …")
            subprocess.run(["rm", "-rf", REPO_DIR], check=True)
        print("Cloning repository …")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        print("✅ Repository cloned.")
    else:
        print("Repository already present — pulling latest changes …")
        result = subprocess.run(
            ["git", "-C", REPO_DIR, "pull", "--ff-only"],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            print(f"⚠️  git pull failed (exit {result.returncode}): {result.stderr.strip()}")
            print("   Continuing with the existing checkout.")
        else:
            print(result.stdout.strip() or "Already up to date.")

    PROJECT_ROOT = REPO_DIR
else:
    # ── Local: notebook lives in <project_root>/ai_model/ ───────────────────
    # __file__ is undefined in Jupyter notebooks; detect root via file probe.
    _cwd = os.getcwd()
    _candidates = [_cwd, os.path.dirname(_cwd)]
    PROJECT_ROOT = next(
        (p for p in _candidates if os.path.exists(os.path.join(p, "indicators.py"))),
        _cwd,
    )

# Make project modules importable
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"\n{'Colab' if IN_COLAB else 'Local'} environment detected.")
print(f"Project root: {PROJECT_ROOT}")

Cloning repository …
✅ Repository cloned.

Colab environment detected.
Project root: /content/pse-trading-bot


### 💾 Optional: Mount Google Drive for Persistent Checkpoints

Run the cell below **only in Colab** if you want checkpoints saved to your Google Drive and preserved across runtime resets.  

After mounting set `USE_DRIVE = True` in the **Configuration** cell.  

Skipping this cell saves checkpoints to `/content/` (lost on runtime reset).

In [2]:
# ── Optional: mount Google Drive ──────────────────────────────────────────
# After mounting, set USE_DRIVE = True in the Configuration cell (Section 1).
DRIVE_MOUNTED = False

_in_colab = "IN_COLAB" in dir() and IN_COLAB
if _in_colab:
    from google.colab import drive as _drive
    try:
        _drive.mount("/content/drive")
        DRIVE_MOUNTED = True
        print("✅ Google Drive mounted at /content/drive")
        print("   Set USE_DRIVE = True in the Configuration cell to persist checkpoints.")
    except Exception as exc:
        _msg = str(exc)
        if "cancelled" in _msg.lower() or "cancel" in _msg.lower():
            print("Drive mount cancelled — checkpoints will be stored in /content/ (ephemeral).")
        elif "authentication" in _msg.lower() or "credentials" in _msg.lower():
            print("Drive authentication failed. Re-run this cell and complete the sign-in prompt.")
        else:
            print(f"Drive mount skipped ({type(exc).__name__}: {_msg}).")
            print("Checkpoints will be stored in /content/ (lost on runtime reset).")
else:
    print("Not running in Colab — Drive mount skipped.")

Mounted at /content/drive
✅ Google Drive mounted at /content/drive
   Set USE_DRIVE = True in the Configuration cell to persist checkpoints.


## 0 · Install dependencies

Run once to make sure all packages are available.

In [3]:
# ── Install Python dependencies ────────────────────────────────────────────
# Active by default so it works on both Colab and local environments.
# Packages already at the required version are skipped automatically by pip.
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "yfinance>=0.2.0", "pandas>=2.0.0", "numpy>=1.24.0",
    "matplotlib>=3.7.0", "plotly>=5.22.0",
    "torch>=2.6.0", "scikit-learn>=1.3.0", "scipy>=1.10.0", "tqdm>=4.0.0",
])
print("✅ All dependencies installed.")

# Colab sometimes requires a runtime restart after first-time package installs
try:
    import google.colab  # noqa: F401
    print("\n⚠️  Colab tip: if you just installed packages for the first time,")
    print("   go to Runtime → Restart session, then re-run all cells from the top.")
except ImportError:
    pass

✅ All dependencies installed.

⚠️  Colab tip: if you just installed packages for the first time,
   go to Runtime → Restart session, then re-run all cells from the top.


## 1 · ⚙️ Configuration

**Edit this cell** to choose your tickers and training window. Everything else in the notebook adapts automatically.

| Variable | Description | Example |
|----------|-------------|---------|
| `TICKERS` | Yahoo Finance PSE symbols | `["BDO.PS", "SM.PS"]` |
| `START_DATE` | Training start date (inclusive) | `"2025-02-01"` |
| `END_DATE` | Training end date (inclusive) | `"2025-02-28"` |
| `INTERVAL` | Candle interval | `"1d"` (daily) or `"1h"` (hourly) |

> **Note on 1-minute data:** Yahoo Finance limits `interval="1m"` to the **last 30 days**. For longer history use `"5m"`, `"1h"`, or `"1d"`.

In [9]:
# ============================================================
# ⚙️  CONFIGURATION — edit this cell to customise the run
# ============================================================

# Target PSE tickers (Yahoo Finance format)
TICKERS = [
    #"BTC-USD",
    "ETH-USD"
    # "SOL-USD"
]

# Date range for data collection (YYYY-MM-DD strings)
START_DATE = "2026-01-07"   # inclusive start date
END_DATE   = "2026-03-07"   # inclusive end date

# Candle interval — choose based on date range:
#   "1m"  → intraday 1-minute  (Yahoo Finance limit: last 30 days only)
#   "5m"  → intraday 5-minute  (Yahoo Finance limit: last 60 days only)
#   "1h"  → hourly              (Yahoo Finance limit: last 730 days)
#   "1d"  → daily               (no limit — recommended for multi-month history)
INTERVAL = "1h"

# ── Model hyperparameters ───────────────────────────────────────────────────
SEQ_LEN         = 96     # look-back window (candles)
LABEL_HORIZON   = 4      # predict N candles ahead
LABEL_THRESHOLD = 0.004  # ±0.2 % return threshold for BUY / SELL

D_MODEL            = 64   # Transformer embedding dimension
NHEAD              = 4    # number of attention heads
NUM_ENCODER_LAYERS = 2    # stacked encoder layers
DIM_FEEDFORWARD    = 128  # feed-forward inner dimension
DROPOUT            = 0.1

BATCH_SIZE    = 64
NUM_EPOCHS    = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
PATIENCE      = 20        # early-stopping patience (epochs)
VAL_SPLIT     = 0.15
TEST_SPLIT    = 0.15
RANDOM_SEED   = 42

# ── Output paths ────────────────────────────────────────────────────────────
import os, sys

# PROJECT_ROOT is set by the Colab Setup cell (Section 0).
# If you skipped that cell, fall back to cwd-based auto-detection.
if "PROJECT_ROOT" not in dir():
    _cwd = os.getcwd()
    _candidates = [_cwd, os.path.dirname(_cwd)]
    PROJECT_ROOT = next(
        (p for p in _candidates if os.path.exists(os.path.join(p, "indicators.py"))),
        _cwd,
    )
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)

# ── Checkpoint location ─────────────────────────────────────────────────────
# Set USE_DRIVE = True (after running the Drive mount cell) to save
# checkpoints to Google Drive and keep them across Colab session resets.
USE_DRIVE = False

if USE_DRIVE and os.path.isdir("/content/drive/MyDrive"):
    CHECKPOINT_DIR = "/content/drive/MyDrive/pse_bot_checkpoints"
else:
    CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "ai_model", "checkpoints")

BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── GPU info ─────────────────────────────────────────────────────────────────
import torch
_dev = (
    "cuda" if torch.cuda.is_available()
    else ("mps" if (hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
          else "cpu")
)
if _dev == "cuda":
    _gpu_name = torch.cuda.get_device_name(0)
    _gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🚀 GPU: {_gpu_name}  ({_gpu_mem:.1f} GB VRAM)")
elif _dev == "mps":
    print("🚀 Apple Silicon GPU (MPS) detected.")
else:
    print("⚠️  No GPU found — training will use CPU (slower).")
    try:
        import google.colab  # noqa: F401
        print("   Tip: Runtime → Change runtime type → T4 GPU")
    except ImportError:
        pass

print("\nConfiguration loaded.")
print(f"  Tickers   : {TICKERS}")
print(f"  Date range: {START_DATE} → {END_DATE}")
print(f"  Interval  : {INTERVAL}")
print(f"  seq_len   : {SEQ_LEN},  label_horizon: {LABEL_HORIZON}")
print(f"  Checkpoint: {BEST_MODEL_PATH}")

🚀 GPU: Tesla T4  (15.6 GB VRAM)

Configuration loaded.
  Tickers   : ['ETH-USD']
  Date range: 2026-01-07 → 2026-03-07
  Interval  : 1h
  seq_len   : 96,  label_horizon: 4
  Checkpoint: /content/pse-trading-bot/ai_model/checkpoints/best_model.pt


## 2 · 📥 Data Collection with yfinance

We download OHLCV data for every configured ticker using an **explicit start / end date range**. All tickers are concatenated into a single long-format DataFrame with a `Ticker` column.

In [10]:
from datetime import date as _date, timedelta
import pandas as pd
import yfinance as yf

def fetch_ohlcv(ticker: str, start: str, end: str, interval: str) -> pd.DataFrame | None:
    """Download OHLCV for one ticker via yfinance.

    yfinance's ``end`` is exclusive, so we add one day to include END_DATE.
    """
    end_exclusive = (_date.fromisoformat(end) + timedelta(days=1)).strftime("%Y-%m-%d")
    raw = yf.download(
        tickers=ticker,
        start=start,
        end=end_exclusive,
        interval=interval,
        auto_adjust=True,
        progress=False,
    )
    if raw is None or raw.empty:
        print(f"  ⚠️  No data for {ticker}")
        return None

    # Flatten MultiIndex columns that yfinance may produce
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    cols = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in raw.columns]
    df = raw[cols].copy()
    df.index.name = "Datetime"
    df = df.reset_index()
    df["Ticker"] = ticker
    df["Datetime"] = pd.to_datetime(df["Datetime"]).dt.tz_localize(None)
    return df


# ── Download all configured tickers ────────────────────────────────────────
frames = []
for ticker in TICKERS:
    print(f"Fetching {ticker} …", end=" ")
    df = fetch_ohlcv(ticker, START_DATE, END_DATE, INTERVAL)
    if df is not None and not df.empty:
        frames.append(df)
        print(f"{len(df):,} rows")

if not frames:
    raise RuntimeError("No data fetched. Check your tickers and date range.")

raw_df = pd.concat(frames, ignore_index=True)
raw_df.sort_values(["Datetime", "Ticker"], inplace=True)
raw_df.reset_index(drop=True, inplace=True)

print(f"\nTotal rows : {len(raw_df):,}")
print(f"Tickers    : {raw_df['Ticker'].nunique()}")
print(f"Date range : {raw_df['Datetime'].min()} → {raw_df['Datetime'].max()}")
raw_df.head()

Fetching ETH-USD … 1,440 rows

Total rows : 1,440
Tickers    : 1
Date range : 2026-01-07 00:00:00 → 2026-03-07 23:00:00


Price,Datetime,Open,High,Low,Close,Volume,Ticker
0,2026-01-07 00:00:00,3292.264648,3292.643555,3263.868896,3263.868896,0,ETH-USD
1,2026-01-07 01:00:00,3256.200439,3269.447510,3238.558594,3243.206299,0,ETH-USD
2,2026-01-07 02:00:00,3243.776123,3269.655762,3238.649902,3267.566895,279083008,ETH-USD
3,2026-01-07 03:00:00,3268.030273,3277.964355,3263.568848,3264.334717,231419904,ETH-USD
4,2026-01-07 04:00:00,3264.212646,3268.595459,3253.978760,3255.651855,0,ETH-USD


## 3 · 🔍 Data Quality Checks

Before training we verify:
- **Missing candles** — gaps larger than expected for the chosen interval.
- **Zero-volume rows** — may indicate suspended trading.
- **Price outliers** — spikes beyond 5 σ from the rolling mean.

In [11]:
import numpy as np

quality_report = []

for ticker, grp in raw_df.groupby("Ticker"):
    grp = grp.sort_values("Datetime").reset_index(drop=True)
    n_rows       = len(grp)
    n_zero_vol   = int((grp["Volume"] == 0).sum())
    n_null_close = int(grp["Close"].isna().sum())

    # Outlier detection: close price deviating > 5σ from 20-bar rolling mean
    roll_mean = grp["Close"].rolling(20, min_periods=1).mean()
    roll_std  = grp["Close"].rolling(20, min_periods=1).std().fillna(0)
    outliers  = int(((grp["Close"] - roll_mean).abs() > 5 * roll_std).sum())

    quality_report.append({
        "Ticker":       ticker,
        "Rows":         n_rows,
        "Zero Vol":     n_zero_vol,
        "Null Close":   n_null_close,
        "Outliers (5σ)": outliers,
    })

qdf = pd.DataFrame(quality_report)
print(qdf.to_string(index=False))

# Flag any ticker with > 5 % zero-volume candles
for _, row in qdf.iterrows():
    if row["Rows"] > 0 and row["Zero Vol"] / row["Rows"] > 0.05:
        print(f"\n⚠️  {row['Ticker']}: {row['Zero Vol'] / row['Rows']:.1%} zero-volume candles.")

print("\n✅ Quality check complete.")

 Ticker  Rows  Zero Vol  Null Close  Outliers (5σ)
ETH-USD  1440       751           0              0

⚠️  ETH-USD: 52.2% zero-volume candles.

✅ Quality check complete.


## 4 · 📊 Price Visualisation

Interactive close-price chart for each ticker over the selected date range.

In [12]:
import plotly.express as px

fig = px.line(
    raw_df,
    x="Datetime",
    y="Close",
    color="Ticker",
    title=f"PSE Close Prices  ({START_DATE} → {END_DATE}, interval={INTERVAL})",
    labels={"Close": "Close Price (PHP)", "Datetime": "Date"},
)
fig.update_layout(hovermode="x unified", legend_title_text="Ticker")
fig.show()

## 5 · 🔧 Feature Engineering

We call `indicators.add_indicators()` — the same function used by the live trading pipeline — to compute **13 features** per candle:

| Group | Features |
|-------|----------|
| Price | Open, High, Low, Close |
| Volume | Volume |
| Trend | EMA_9, EMA_21 |
| Momentum | RSI |
| Volatility | BB_upper, BB_middle, BB_lower, Returns, Volatility |

In [13]:
import indicators   # project module at repo root

processed_df = indicators.add_indicators(raw_df.copy())
print(f"Processed shape: {processed_df.shape}")
print("Columns:", list(processed_df.columns))
processed_df.head()

Processed shape: (1440, 15)
Columns: ['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'EMA_9', 'EMA_21', 'RSI', 'BB_upper', 'BB_middle', 'BB_lower', 'Returns', 'Volatility']


Price,Datetime,Open,High,Low,Close,Volume,Ticker,EMA_9,EMA_21,RSI,BB_upper,BB_middle,BB_lower,Returns,Volatility
0,2026-01-07 00:00:00,3292.264648,3292.643555,3263.868896,3263.868896,0,ETH-USD,3263.868896,3263.868896,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-01-07 01:00:00,3256.200439,3269.447510,3238.558594,3243.206299,0,ETH-USD,3259.736377,3261.990479,0.000000,NaN,NaN,NaN,-0.006331,NaN
2,2026-01-07 02:00:00,3243.776123,3269.655762,3238.649902,3267.566895,279083008,ETH-USD,3261.302480,3262.497425,8.314924,NaN,NaN,NaN,0.007511,NaN
3,2026-01-07 03:00:00,3268.030273,3277.964355,3263.568848,3264.334717,231419904,ETH-USD,3261.908928,3262.664452,8.217295,NaN,NaN,NaN,-0.000989,NaN
4,2026-01-07 04:00:00,3264.212646,3268.595459,3253.978760,3255.651855,0,ETH-USD,3260.657513,3262.026943,7.947338,NaN,NaN,NaN,-0.002660,NaN


In [14]:
# ── Spot-check: visualise EMA crossover on the first ticker ────────────────
ticker_sample = TICKERS[0]
sample = processed_df[processed_df["Ticker"] == ticker_sample].copy()

import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=sample["Datetime"], y=sample["Close"],
                         name="Close", line=dict(color="steelblue")))
fig.add_trace(go.Scatter(x=sample["Datetime"], y=sample[f"EMA_9"],
                         name="EMA 9", line=dict(color="orange", dash="dot")))
fig.add_trace(go.Scatter(x=sample["Datetime"], y=sample[f"EMA_21"],
                         name="EMA 21", line=dict(color="red", dash="dash")))
fig.update_layout(
    title=f"{ticker_sample} — Close with EMA Crossover",
    xaxis_title="Date",
    yaxis_title="Price (PHP)",
    hovermode="x unified",
)
fig.show()

## 6 · 🏷️ Label Design & Class-Imbalance Analysis

Labels are computed from the **look-ahead return** over `LABEL_HORIZON` candles:

```
future_return = (Close[t + horizon] - Close[t]) / Close[t]

BUY  (0)  if future_return ≥ +LABEL_THRESHOLD   (default: +0.2 %)
HOLD (1)  if |future_return| < LABEL_THRESHOLD
SELL (2)  if future_return ≤ -LABEL_THRESHOLD   (default: -0.2 %)
```

On 1-minute PSE data, **HOLD typically represents 80–90 %** of all labels.
The training loop counters this with inverse-frequency class weights.

In [15]:
from ai_model.config   import ModelConfig
from ai_model.features import build_feature_pipeline, split_data

cfg = ModelConfig(
    seq_len         = SEQ_LEN,
    label_horizon   = LABEL_HORIZON,
    label_threshold = LABEL_THRESHOLD,
    d_model            = D_MODEL,
    nhead              = NHEAD,
    num_encoder_layers = NUM_ENCODER_LAYERS,
    dim_feedforward    = DIM_FEEDFORWARD,
    dropout            = DROPOUT,
    batch_size         = BATCH_SIZE,
    num_epochs         = NUM_EPOCHS,
    learning_rate      = LEARNING_RATE,
    weight_decay       = WEIGHT_DECAY,
    patience           = PATIENCE,
    val_split          = VAL_SPLIT,
    test_split         = TEST_SPLIT,
    random_seed        = RANDOM_SEED,
    checkpoint_dir     = CHECKPOINT_DIR,
    best_model_path    = BEST_MODEL_PATH,
)

featured_df, scaler_stats = build_feature_pipeline(processed_df.copy(), cfg)
print(f"Feature pipeline output shape: {featured_df.shape}")
print(f"Label distribution (raw counts):")

label_map = {0: "BUY", 1: "HOLD", 2: "SELL"}
dist = featured_df["Label"].value_counts().sort_index()
for k, v in dist.items():
    print(f"  {label_map[int(k)]:<5} ({int(k)}): {v:>6,}  ({v / len(featured_df):.1%})")

Feature pipeline output shape: (1416, 16)
Label distribution (raw counts):
  BUY   (0):    420  (29.7%)
  HOLD  (1):    498  (35.2%)
  SELL  (2):    498  (35.2%)


In [16]:
# ── Bar chart of label distribution ────────────────────────────────────────
import plotly.express as px

dist_df = (
    featured_df["Label"]
    .value_counts()
    .reset_index()
    .rename(columns={"Label": "count", "index": "label_int"})
    .sort_values("Label")
)
dist_df["Signal"] = dist_df["Label"].map(label_map)

fig = px.bar(
    dist_df,
    x="Signal",
    y="count",
    color="Signal",
    color_discrete_map={"BUY": "green", "HOLD": "gray", "SELL": "red"},
    title=f"Label Distribution  (horizon={LABEL_HORIZON}, threshold={LABEL_THRESHOLD:.3f})",
    text_auto=True,
)
fig.update_layout(showlegend=False, xaxis_title="Signal", yaxis_title="Count")
fig.show()

KeyError: 'Label'

## 7 · 🗂️ Dataset Construction

`TradingSequenceDataset` slides a window of length `SEQ_LEN` over each ticker's normalised feature matrix to produce overlapping `(SEQ_LEN, num_features)` tensors paired with a scalar label.

In [17]:
from ai_model.dataset import TradingSequenceDataset, compute_class_weights, build_dataloaders

train_df, val_df, test_df = split_data(featured_df, cfg)

train_ds = TradingSequenceDataset(train_df, cfg)
val_ds   = TradingSequenceDataset(val_df,   cfg)
test_ds  = TradingSequenceDataset(test_df,  cfg)

print(f"Train sequences : {len(train_ds):,}")
print(f"Val   sequences : {len(val_ds):,}")
print(f"Test  sequences : {len(test_ds):,}")
print()

class_weights = compute_class_weights(train_ds)
print("Class weights:")
for i, w in enumerate(class_weights.tolist()):
    print(f"  {label_map[i]:<5}: {w:.4f}")

# Sample one batch to verify shapes
from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False)

x_sample, y_sample = next(iter(train_loader))
print(f"\nSample batch shapes:  x={tuple(x_sample.shape)}  y={tuple(y_sample.shape)}")

Train sequences : 926
Val   sequences : 85
Test  sequences : 117

Class weights:
  BUY  : 1.1184
  HOLD : 0.9768
  SELL : 0.9242

Sample batch shapes:  x=(64, 96, 13)  y=(64,)


## 8 · 🏗️ Transformer Model Architecture

Architecture (input flows top to bottom):

    Input  (batch, seq_len, num_features)
      │
      ├─ Linear projection → (batch, seq_len, d_model)
      ├─ Sinusoidal Positional Encoding
      ├─ N × TransformerEncoderLayer  (pre-norm, GELU, multi-head attention)
      ├─ Mean pool → (batch, d_model)
      └─ MLP classifier → (batch, 3)   [BUY / HOLD / SELL logits]


In [18]:
import torch
from ai_model.model import TransformerSignalModel

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"Device: {device}")

model = TransformerSignalModel(cfg).to(device)
print(f"Parameters: {model.count_parameters():,}")
print(model)

Device: cuda
Parameters: 72,323
TransformerSignalModel(
  (input_proj): Linear(in_features=13, out_features=64, bias=True)
  (pos_enc): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=Tr

## 9 · 🚀 Training

The training loop uses:
- **AdamW** optimiser with cosine annealing LR schedule
- **Weighted cross-entropy loss** to handle HOLD dominance
- **Gradient clipping** (max norm = 1.0)
- **Early stopping** (patience = `PATIENCE` epochs, monitored on val loss)
- Automatic **checkpoint** of the best epoch

In [19]:
import time
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# tqdm provides a progress bar; falls back gracefully if not installed
try:
    from tqdm.notebook import tqdm as _tqdm
except ImportError:
    def _tqdm(iterable, **_kwargs): return iterable

torch.manual_seed(cfg.random_seed)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.num_epochs, eta_min=1e-6)

history = []
best_val_loss    = float("inf")
patience_counter = 0

for epoch in _tqdm(range(1, cfg.num_epochs + 1), desc="Training", unit="epoch"):
    t0 = time.time()

    # ── Train ──────────────────────────────────────────────────────────
    model.train()
    train_loss = train_correct = train_total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss   = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss    += loss.item() * len(yb)
        train_correct += (logits.argmax(1) == yb).sum().item()
        train_total   += len(yb)

    # ── Validate ───────────────────────────────────────────────────────
    model.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            val_loss    += criterion(logits, yb).item() * len(yb)
            val_correct += (logits.argmax(1) == yb).sum().item()
            val_total   += len(yb)

    scheduler.step()

    avg_train = train_loss / max(train_total, 1)
    avg_val   = val_loss   / max(val_total,   1)
    acc_train = train_correct / max(train_total, 1)
    acc_val   = val_correct   / max(val_total,   1)
    elapsed   = time.time() - t0

    history.append({"epoch": epoch, "train_loss": avg_train, "val_loss": avg_val,
                    "train_acc": acc_train, "val_acc": acc_val})

    print(f"Epoch {epoch:3d}/{cfg.num_epochs} | "
          f"train_loss={avg_train:.4f} acc={acc_train:.3f} | "
          f"val_loss={avg_val:.4f} acc={acc_val:.3f} | {elapsed:.1f}s")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": avg_val,
            "val_acc":  acc_val,
            "cfg": cfg,
        }, cfg.best_model_path)
        print(f"  ✓ Checkpoint saved (val_loss={avg_val:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= cfg.patience:
            print(f"\nEarly stopping at epoch {epoch} "
                  f"(no improvement for {cfg.patience} epochs).")
            break

print(f"\nBest val_loss: {best_val_loss:.4f}")

Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

Epoch   1/50 | train_loss=1.1572 acc=0.354 | val_loss=1.1257 acc=0.341 | 1.2s
  ✓ Checkpoint saved (val_loss=1.1257)
Epoch   2/50 | train_loss=1.0950 acc=0.414 | val_loss=1.1087 acc=0.341 | 0.2s
  ✓ Checkpoint saved (val_loss=1.1087)
Epoch   3/50 | train_loss=1.0795 acc=0.402 | val_loss=1.1321 acc=0.365 | 0.2s
Epoch   4/50 | train_loss=1.0812 acc=0.404 | val_loss=1.0924 acc=0.341 | 0.1s
  ✓ Checkpoint saved (val_loss=1.0924)
Epoch   5/50 | train_loss=1.0563 acc=0.431 | val_loss=1.0906 acc=0.376 | 0.1s
  ✓ Checkpoint saved (val_loss=1.0906)
Epoch   6/50 | train_loss=1.0462 acc=0.426 | val_loss=1.0899 acc=0.341 | 0.1s
  ✓ Checkpoint saved (val_loss=1.0899)
Epoch   7/50 | train_loss=1.0339 acc=0.452 | val_loss=1.1016 acc=0.294 | 0.1s
Epoch   8/50 | train_loss=1.0305 acc=0.450 | val_loss=1.0867 acc=0.341 | 0.1s
  ✓ Checkpoint saved (val_loss=1.0867)
Epoch   9/50 | train_loss=1.0156 acc=0.468 | val_loss=1.0941 acc=0.341 | 0.1s
Epoch  10/50 | train_loss=0.9968 acc=0.497 | val_loss=1.0857 acc

In [ ]:
# ── Download checkpoint to your local machine (Colab only) ─────────────────
# Run this cell after training to save the checkpoint file locally.
# On local Jupyter, prints the path where the file is already stored.
try:
    from google.colab import files as _colab_files
    _ckpt = cfg.best_model_path
    if os.path.exists(_ckpt):
        print(f"Downloading checkpoint: {_ckpt}")
        _colab_files.download(_ckpt)
    else:
        print("No checkpoint found. Run the training cell first.")
except ImportError:
    print(f"Checkpoint saved locally at:\n  {cfg.best_model_path}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10 · 📈 Training Curves

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

hist_df = pd.DataFrame(history)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Loss", "Accuracy"))

fig.add_trace(go.Scatter(x=hist_df["epoch"], y=hist_df["train_loss"],
                         name="Train Loss", line=dict(color="royalblue")), row=1, col=1)
fig.add_trace(go.Scatter(x=hist_df["epoch"], y=hist_df["val_loss"],
                         name="Val Loss",   line=dict(color="orangered")), row=1, col=1)

fig.add_trace(go.Scatter(x=hist_df["epoch"], y=hist_df["train_acc"],
                         name="Train Acc",  line=dict(color="royalblue"),
                         showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=hist_df["epoch"], y=hist_df["val_acc"],
                         name="Val Acc",    line=dict(color="orangered"),
                         showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Loss",     row=1, col=1)
fig.update_yaxes(title_text="Accuracy", row=1, col=2)
fig.update_layout(title="Training History", hovermode="x unified")
fig.show()

## 11 · 🎯 Evaluation

We reload the best checkpoint and run the test set through:
1. **Classification metrics** — per-class precision / recall / F1
2. **Trading simulation** — Sharpe ratio, max drawdown, win rate

In [ ]:
from ai_model.evaluate import evaluate_model, print_report

# Reload best checkpoint weights
ckpt = torch.load(cfg.best_model_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} "
      f"(val_loss={ckpt['val_loss']:.4f}, val_acc={ckpt['val_acc']:.3f})")

# Get close prices aligned with the test sequences for trading simulation
test_closes = (
    test_df.sort_values("Datetime")["Close"]
    .iloc[cfg.seq_len:]          # shift to align with labels
    .reset_index(drop=True)
    .to_numpy()
)

report = evaluate_model(model, test_loader, cfg, device, close_prices=test_closes)
print_report(report)

Loaded checkpoint from epoch 4 (val_loss=1.0905, val_acc=0.469)

  CLASSIFICATION METRICS
  Overall Accuracy : 28.14%

  Class    Precision    Recall        F1   Support
  ------------------------------------------------
  BUY         0.2450    0.3854    0.2996       192
  HOLD        0.0000    0.0000    0.0000       356
  SELL        0.3055    0.6651    0.4187       209

  TRADING METRICS (long-only simulation)
  Sharpe Ratio     : 6.3901
  Max Drawdown     : -9.59%
  Win Rate         : 50.00%
  # Trades         : 10



In [ ]:
# ── Visualise confusion matrix ─────────────────────────────────────────────
import plotly.figure_factory as ff

cm    = report["classification"]["confusion_matrix"]
labels = ["BUY", "HOLD", "SELL"]

fig = ff.create_annotated_heatmap(
    z        = cm,
    x        = [f"Pred {l}" for l in labels],
    y        = [f"True {l}" for l in labels],
    colorscale = "Blues",
    showscale  = True,
)
fig.update_layout(title="Confusion Matrix (Test Set)")
fig.show()

## 12 · 🤝 Integration with TradingAgent

`TransformerStrategy` is a drop-in `BaseStrategy` subclass.  Pass it to `TradingAgent` exactly like any rule-based strategy.

In [ ]:
from ai_model.predict import TransformerPredictor, TransformerStrategy
from trading_agent    import TradingAgent
from portfolio        import Portfolio

# Load predictor from checkpoint
predictor = TransformerPredictor.from_checkpoint(cfg.best_model_path, device=device)

# ── Quick single-ticker prediction ─────────────────────────────────────────
ticker_test = TICKERS[0]
signal, probs = predictor.predict_from_df(
    df           = featured_df,
    ticker       = ticker_test,
    scaler_stats = scaler_stats,
)
print(f"Latest signal for {ticker_test}: {signal}")
print(f"  P(BUY)={probs[0]:.3f}  P(HOLD)={probs[1]:.3f}  P(SELL)={probs[2]:.3f}")

Latest signal for ETH-USD: BUY
  P(BUY)=0.553  P(HOLD)=0.140  P(SELL)=0.307


In [ ]:
# ── Full agent run on the test window ──────────────────────────────────────
strategy = TransformerStrategy(
    checkpoint_path      = cfg.best_model_path,
    scaler_stats         = scaler_stats,
    confidence_threshold = 0.0,   # only act when model is > 50 % confident
)
strategy.set_dataframe(featured_df)   # give strategy access to full history

portfolio = Portfolio(initial_capital=1_000_000.0)
agent     = TradingAgent(portfolio=portfolio, strategy=strategy)
signals_df = agent.run(test_df.copy())

print("\nSignal counts in the test window:")
print(signals_df["Signal"].value_counts().to_string())
print()
print(portfolio.summary())


Signal counts in the test window:
Signal
BUY     655
SELL    139
HOLD     59

{'cash': 1000000.0, 'initial_capital': 1000000.0, 'total_realized_pnl': 0, 'daily_realized_pnl': 0.0, 'open_positions': {}, 'unrealized_pnl': {}, 'market_value': 1000000.0, 'total_return_pct': 0.0}


## 13 · 🔬 Hyperparameter Tuning Template

The cell below shows a **manual mini-grid search** over key hyperparameters. For larger searches, replace the loop with [Optuna](https://optuna.org/) (`pip install optuna`).

In [ ]:
# ── Mini grid search — edit the lists below to expand the search ───────────
# WARNING: this will retrain the model for every combination; may take a while.

PARAM_GRID = {
    "seq_len":         [15, 30],
    "label_horizon":   [5, 10],
    "label_threshold": [0.002, 0.003],
}

import itertools

grid_results = []
keys   = list(PARAM_GRID.keys())
combos = list(itertools.product(*PARAM_GRID.values()))

print(f"Running {len(combos)} hyperparameter combinations …\n")

for combo in combos:
    params = dict(zip(keys, combo))
    print(f"  Testing: {params} … ", end="")

    # Build a fresh config with this combo
    run_cfg = ModelConfig(
        seq_len         = params["seq_len"],
        label_horizon   = params["label_horizon"],
        label_threshold = params["label_threshold"],
        d_model            = D_MODEL,
        nhead              = NHEAD,
        num_encoder_layers = NUM_ENCODER_LAYERS,
        dim_feedforward    = DIM_FEEDFORWARD,
        dropout            = DROPOUT,
        batch_size         = BATCH_SIZE,
        num_epochs         = 10,          # short run for grid search
        learning_rate      = LEARNING_RATE,
        weight_decay       = WEIGHT_DECAY,
        patience           = 5,
        val_split          = VAL_SPLIT,
        test_split         = TEST_SPLIT,
        random_seed        = RANDOM_SEED,
        checkpoint_dir     = CHECKPOINT_DIR,
        best_model_path    = os.path.join(CHECKPOINT_DIR, "grid_best.pt"),
    )

    # Feature pipeline for this config
    run_featured, _ = build_feature_pipeline(processed_df.copy(), run_cfg)
    run_train, run_val, run_test = split_data(run_featured, run_cfg)

    run_train_ds = TradingSequenceDataset(run_train, run_cfg)
    run_val_ds   = TradingSequenceDataset(run_val,   run_cfg)
    if len(run_train_ds) == 0 or len(run_val_ds) == 0:
        print("skipped (insufficient sequences)")
        continue

    run_weights = compute_class_weights(run_train_ds)
    run_loader  = DataLoader(run_train_ds, batch_size=run_cfg.batch_size, shuffle=True, drop_last=True)
    run_val_loader = DataLoader(run_val_ds, batch_size=run_cfg.batch_size)

    run_model  = TransformerSignalModel(run_cfg).to(device)
    run_crit   = nn.CrossEntropyLoss(weight=run_weights.to(device))
    run_optim  = AdamW(run_model.parameters(), lr=run_cfg.learning_rate, weight_decay=run_cfg.weight_decay)
    run_sched  = CosineAnnealingLR(run_optim, T_max=run_cfg.num_epochs, eta_min=1e-6)

    best_vl = float("inf")
    for ep in range(1, run_cfg.num_epochs + 1):
        run_model.train()
        for xb, yb in run_loader:
            xb, yb = xb.to(device), yb.to(device)
            run_optim.zero_grad()
            nn.utils.clip_grad_norm_(run_model.parameters(), 1.0)
            run_crit(run_model(xb), yb).backward()
            run_optim.step()
        run_sched.step()

        run_model.eval()
        vl = 0.0
        with torch.no_grad():
            for xb, yb in run_val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vl += run_crit(run_model(xb), yb).item() * len(yb)
        vl /= max(len(run_val_ds), 1)
        if vl < best_vl:
            best_vl = vl

    print(f"best_val_loss={best_vl:.4f}")
    grid_results.append({**params, "best_val_loss": best_vl})

grid_df = pd.DataFrame(grid_results).sort_values("best_val_loss")
print("\nGrid search results (sorted by val loss):")
print(grid_df.to_string(index=False))

Running 8 hyperparameter combinations …

  Testing: {'seq_len': 15, 'label_horizon': 5, 'label_threshold': 0.002} … best_val_loss=1.1045
  Testing: {'seq_len': 15, 'label_horizon': 5, 'label_threshold': 0.003} … best_val_loss=1.0976
  Testing: {'seq_len': 15, 'label_horizon': 10, 'label_threshold': 0.002} … best_val_loss=1.0871
  Testing: {'seq_len': 15, 'label_horizon': 10, 'label_threshold': 0.003} … best_val_loss=1.1020
  Testing: {'seq_len': 30, 'label_horizon': 5, 'label_threshold': 0.002} … best_val_loss=1.0964
  Testing: {'seq_len': 30, 'label_horizon': 5, 'label_threshold': 0.003} … best_val_loss=1.1049
  Testing: {'seq_len': 30, 'label_horizon': 10, 'label_threshold': 0.002} … best_val_loss=1.0947
  Testing: {'seq_len': 30, 'label_horizon': 10, 'label_threshold': 0.003} … best_val_loss=1.1157

Grid search results (sorted by val loss):
 seq_len  label_horizon  label_threshold  best_val_loss
      15             10            0.002       1.087058
      30             10         

## 14 · 🚢 Deployment — TorchScript Export & Checklist

Export the trained model to TorchScript for faster portable inference (no Python source required at inference time).

In [ ]:
# ── Reload best model ──────────────────────────────────────────────────────
ckpt = torch.load(cfg.best_model_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# ── TorchScript export ─────────────────────────────────────────────────────
scripted_path = os.path.join(CHECKPOINT_DIR, "model_scripted.pt")
scripted_model = torch.jit.script(model)
scripted_model.save(scripted_path)
print(f"TorchScript model saved to: {scripted_path}")

# ── Inference latency benchmark ────────────────────────────────────────────
dummy_input = torch.randn(1, cfg.seq_len, cfg.num_features).to(device)
# Warm-up
for _ in range(10):
    with torch.inference_mode():
        scripted_model(dummy_input)

import time
N = 200
t0 = time.perf_counter()
for _ in range(N):
    with torch.inference_mode():
        scripted_model(dummy_input)
elapsed_ms = (time.perf_counter() - t0) / N * 1000
print(f"Average inference latency: {elapsed_ms:.2f} ms / prediction")
print(f"Target: < 50 ms  →  {'✅ OK' if elapsed_ms < 50 else '⚠️  Consider smaller model'}")

### Deployment checklist

- [ ] Retrain on latest 60–90 days of data monthly (model drift).
- [ ] Run in **shadow mode** first: log signals without executing trades.
- [ ] Monitor per-class F1 on held-out recent data as a drift indicator.
- [ ] Register `TransformerStrategy` in `STRATEGY_REGISTRY` in `trading_agent.py`.
- [ ] Run `python main.py --live` to generate real-time signals every minute.